# PhiSat-2 and Sentinel-2 L1C Coupling

This notebook starts from the PhiSat-2 catalogue workflow in notebook 10 and builds product pairs for the names in `notebooks/phisat_list.csv`.

The workflow is:
1. resolve each PhiSat-2 product name through `phidown` to get its sensing time and `GeoFootprint`;
2. search Sentinel-2 L1C products over the PhiSat-2 footprint in a week-scale time window;
3. keep only Sentinel-2 candidates with catalogue `cloudCover < 0.05` and at least 80% coverage of the PhiSat-2 footprint;
4. build `coupling_gdf`, a GeoDataFrame with the paired product names, times, time delta, overlap, cloud cover, and footprints.

Intermediate catalogue responses are cached under `notebooks/data/phisat2_s2_l1c_coupling/` so a full list run can be resumed.


## Credentials and runtime

PhiSat-2 metadata lookup uses the same `[phisat2]` block in `.s5cfg` as notebook 10. Sentinel-2 catalogue search uses the public CDSE OData endpoint through `CopernicusDataSearcher`.

This notebook expects `geopandas`, `shapely`, `pandas`, `tqdm`, and the local `phidown` package. The Sentinel-2 gate keeps only catalogue `cloudCover < 0.05` products; if downstream work needs pixel-level cloud-free confirmation over the exact PhiSat-2 footprint, validate the selected products with the Sentinel-2 cloud masks after download.


In [ ]:
from __future__ import annotations

import configparser
import json
import re
import sys
import time
from pathlib import Path
from typing import Any, Optional, Tuple

import geopandas as gpd
import pandas as pd
from IPython.display import display
from shapely import wkt as shapely_wkt
from shapely.geometry import mapping, shape
try:
    from shapely.validation import make_valid
except ImportError:  # shapely < 2
    make_valid = None
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in [cwd, *cwd.parents] if (candidate / "phidown").exists()),
    cwd,
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import phidown
from phidown.search import CopernicusDataSearcher

print(f"Repo root: {repo_root}")
print(f"phidown version: {phidown.__version__}")


In [ ]:
# Notebook parameters
config_candidates = [repo_root / ".s5cfg", cwd / ".s5cfg"]
CONFIG_FILE = next((path for path in config_candidates if path.exists()), repo_root / ".s5cfg")

PHISAT_LIST = repo_root / "notebooks" / "phisat_list.csv"
OUTPUT_DIR = repo_root / "notebooks" / "data" / "phisat2_s2_l1c_coupling"
PHISAT_METADATA_CACHE = OUTPUT_DIR / "phisat2_metadata.jsonl"
PHISAT_METADATA_ERRORS = OUTPUT_DIR / "phisat2_metadata_errors.csv"
S2_CANDIDATES_CACHE = OUTPUT_DIR / "s2_l1c_candidates.jsonl"
S2_CANDIDATE_ERRORS = OUTPUT_DIR / "s2_l1c_candidate_errors.csv"
MATCH_LOG_CSV = OUTPUT_DIR / "match_log.csv"
COUPLING_GEOJSON = OUTPUT_DIR / "phisat2_s2_l1c_coupling.geojson"
COUPLING_CSV = OUTPUT_DIR / "phisat2_s2_l1c_coupling.csv"

PHISAT_PRODUCT_TYPE = "L1"
PHISAT_RESULTS_PER_NAME = 5
S2_COLLECTION_NAME = "SENTINEL-2"
S2_PRODUCT_TYPE = "S2MSI1C"
S2_RESULTS_PER_PRODUCT = 200
S2_CLOUD_COVER_THRESHOLD = 0.05  # phidown and the final gate use cloudCover < threshold.
MIN_SPATIAL_OVERLAP = 0.80
MAX_TIME_DELTA_DAYS = 21
REQUEST_SLEEP_SECONDS = 0.0

MAX_PRODUCTS = None  # Set to a small integer for a smoke test; None processes the full CSV.
REFRESH_PHISAT_METADATA = False
REFRESH_S2_CANDIDATES = False
DISPLAY_ROWS = 10

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PHISAT_LIST={PHISAT_LIST}")
print(f"CONFIG_FILE={CONFIG_FILE}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"S2_PRODUCT_TYPE={S2_PRODUCT_TYPE}")
print(f"S2_CLOUD_COVER_THRESHOLD={S2_CLOUD_COVER_THRESHOLD}")
print(f"MIN_SPATIAL_OVERLAP={MIN_SPATIAL_OVERLAP:.0%}")
print(f"MAX_TIME_DELTA_DAYS={MAX_TIME_DELTA_DAYS}")


In [ ]:
def has_phisat2_credentials(config_path: Path) -> bool:
    if not config_path.exists():
        return False
    parser = configparser.ConfigParser()
    parser.read(config_path)
    if not parser.has_section("phisat2"):
        return False
    username = parser.get("phisat2", "username", fallback="").strip().strip("'\"")
    password = parser.get("phisat2", "password", fallback="").strip().strip("'\"")
    return bool(username and password)


def load_phisat_product_names(path: Path, max_products: Optional[int] = None) -> pd.Series:
    if not path.exists():
        raise FileNotFoundError(f"PhiSat-2 product list not found: {path}")
    names = pd.read_csv(path, header=None, names=["phisat2_list_name"], dtype=str)["phisat2_list_name"]
    names = names.dropna().str.strip()
    names = names[names.ne("")].drop_duplicates().reset_index(drop=True)
    if max_products is not None:
        names = names.head(max_products).reset_index(drop=True)
    return names


if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_FILE}")
if not has_phisat2_credentials(CONFIG_FILE):
    raise ValueError(f"Config file exists but is missing usable [phisat2] username/password values: {CONFIG_FILE}")

product_names = load_phisat_product_names(PHISAT_LIST, MAX_PRODUCTS)
print(f"Loaded {len(product_names)} PhiSat-2 product names")
display(product_names.to_frame().head(DISPLAY_ROWS))


In [ ]:
def normalize_product_name(value: Any) -> str:
    text = str(value or "").strip().rstrip("/")
    text = Path(text).name
    for suffix in (".zip", ".ZIP", ".SAFE", ".safe"):
        if text.endswith(suffix):
            text = text[: -len(suffix)]
    return text


def _valid_geometry(geom):
    if geom.is_valid:
        return geom
    if make_valid is not None:
        return make_valid(geom)
    return geom.buffer(0)


def parse_footprint(geo_footprint: Any):
    if geo_footprint is None:
        raise ValueError("Missing GeoFootprint")
    if isinstance(geo_footprint, float) and pd.isna(geo_footprint):
        raise ValueError("Missing GeoFootprint")
    if isinstance(geo_footprint, dict):
        geometry = geo_footprint.get("geometry") if geo_footprint.get("type") == "Feature" else geo_footprint
        return _valid_geometry(shape(geometry))
    if isinstance(geo_footprint, str):
        text = geo_footprint.strip()
        if text.startswith("{") or text.startswith("["):
            return parse_footprint(json.loads(text))
        return _valid_geometry(shapely_wkt.loads(text))
    return _valid_geometry(shape(geo_footprint))


def parse_optional_footprint(geo_footprint: Any):
    try:
        return parse_footprint(geo_footprint)
    except Exception:
        return None


def geometry_to_geojson(geom):
    if geom is None or geom.is_empty:
        return None
    return mapping(geom)


def content_time(content_date: Any, key: str = "Start") -> pd.Timestamp:
    if isinstance(content_date, dict):
        value = content_date.get(key) or content_date.get(key.lower())
    elif isinstance(content_date, str) and content_date.strip().startswith("{"):
        return content_time(json.loads(content_date), key)
    else:
        value = content_date
    return pd.to_datetime(value, utc=True)


def iso_z(timestamp: pd.Timestamp) -> str:
    timestamp = pd.to_datetime(timestamp, utc=True)
    return timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")


def time_window(target_time: pd.Timestamp, days: int) -> Tuple[str, str]:
    delta = pd.Timedelta(days=days)
    return iso_z(target_time - delta), iso_z(target_time + delta)


def attribute_value(row: pd.Series, name: str):
    attrs = row.get("Attributes")
    if isinstance(attrs, str) and attrs.strip().startswith("["):
        attrs = json.loads(attrs)
    if not isinstance(attrs, list):
        return None
    for attr in attrs:
        if not isinstance(attr, dict) or attr.get("Name") != name:
            continue
        if "Value" in attr:
            return attr["Value"]
        for value in attr.values():
            if isinstance(value, dict) and "Value" in value:
                return value["Value"]
    return None


def overlap_fraction(reference_geom, candidate_geom) -> float:
    if reference_geom.is_empty or candidate_geom.is_empty:
        return 0.0
    geoms = gpd.GeoSeries([reference_geom, candidate_geom], crs="EPSG:4326")
    try:
        metric_geoms = geoms.to_crs(geoms.estimate_utm_crs())
    except Exception:
        metric_geoms = geoms
    reference_metric = metric_geoms.iloc[0]
    candidate_metric = metric_geoms.iloc[1]
    reference_area = reference_metric.area
    if reference_area == 0:
        return 0.0
    return min(1.0, reference_metric.intersection(candidate_metric).area / reference_area)


def write_jsonl(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, date_format="iso")


def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_json(path, lines=True)


In [ ]:
def phisat2_name_time_window(product_name: str, pad_minutes: int = 5) -> Tuple[str, str]:
    match = re.search(r"_(\d{14})_(\d{14})_[A-Z0-9]+$", normalize_product_name(product_name))
    if match is None:
        raise ValueError(f"Could not parse PhiSat-2 sensing times from product name: {product_name}")
    start = pd.to_datetime(match.group(1), format="%Y%m%d%H%M%S", utc=True)
    end = pd.to_datetime(match.group(2), format="%Y%m%d%H%M%S", utc=True)
    pad = pd.Timedelta(minutes=pad_minutes)
    return iso_z(start - pad), iso_z(end + pad)


def select_phisat2_match(df: pd.DataFrame, product_name: str) -> Tuple[Optional[pd.Series], str]:
    if df.empty or "Name" not in df.columns:
        return None, "not_found"
    wanted = normalize_product_name(product_name)
    normalized = df["Name"].map(normalize_product_name)
    exact = df[normalized.eq(wanted)]
    if len(exact) == 1:
        return exact.iloc[0], "exact"
    if len(exact) > 1:
        return None, "ambiguous_exact"
    if len(df) == 1:
        return df.iloc[0], "single_candidate"
    return None, "ambiguous_candidates"


def query_phisat2_product(product_name: str) -> Tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    searcher = CopernicusDataSearcher()
    start_date, end_date = phisat2_name_time_window(product_name)
    searcher.query_by_filter(
        collection_name="PHISAT-2",
        product_type=PHISAT_PRODUCT_TYPE,
        start_date=start_date,
        end_date=end_date,
        top=PHISAT_RESULTS_PER_NAME,
        config_file=str(CONFIG_FILE),
    )
    df = searcher.execute_query().copy()
    row, status = select_phisat2_match(df, product_name)
    if row is None:
        return None, {
            "phisat2_list_name": product_name,
            "stage": "phisat2_metadata",
            "status": status,
            "candidate_count": len(df),
            "start_date": start_date,
            "end_date": end_date,
        }

    record = row.to_dict()
    record["phisat2_list_name"] = product_name
    record["phisat2_query_status"] = status
    record["phisat2_start_time"] = content_time(record["ContentDate"], "Start").isoformat()
    record["phisat2_end_time"] = content_time(record["ContentDate"], "End").isoformat()
    return record, None


if PHISAT_METADATA_CACHE.exists() and not REFRESH_PHISAT_METADATA:
    phisat_df = read_jsonl(PHISAT_METADATA_CACHE)
    phisat_errors = pd.read_csv(PHISAT_METADATA_ERRORS) if PHISAT_METADATA_ERRORS.exists() else pd.DataFrame()
    print(f"Loaded cached PhiSat-2 metadata rows: {len(phisat_df)}")
else:
    metadata_rows: list[dict[str, Any]] = []
    metadata_errors: list[dict[str, Any]] = []
    for product_name in tqdm(product_names, desc="Resolving PhiSat-2 products"):
        try:
            record, error = query_phisat2_product(product_name)
            if record is not None:
                metadata_rows.append(record)
            if error is not None:
                metadata_errors.append(error)
        except Exception as exc:
            metadata_errors.append({
                "phisat2_list_name": product_name,
                "stage": "phisat2_metadata",
                "status": type(exc).__name__,
                "message": str(exc),
            })
        if REQUEST_SLEEP_SECONDS:
            time.sleep(REQUEST_SLEEP_SECONDS)

    phisat_df = pd.DataFrame(metadata_rows)
    phisat_errors = pd.DataFrame(metadata_errors)
    write_jsonl(phisat_df, PHISAT_METADATA_CACHE)
    phisat_errors.to_csv(PHISAT_METADATA_ERRORS, index=False)

if phisat_df.empty:
    raise LookupError("No PhiSat-2 metadata rows were resolved from the product list")

phisat_df["phisat2_start_time"] = pd.to_datetime(phisat_df["phisat2_start_time"], utc=True)
phisat_df["phisat2_end_time"] = pd.to_datetime(phisat_df["phisat2_end_time"], utc=True)

print(f"Resolved PhiSat-2 metadata rows: {len(phisat_df)}")
print(f"PhiSat-2 metadata errors: {len(phisat_errors)}")
display_columns = [
    "phisat2_list_name",
    "Name",
    "phisat2_start_time",
    "phisat2_end_time",
    "phisat2_query_status",
]
display(phisat_df[[column for column in display_columns if column in phisat_df.columns]].head(DISPLAY_ROWS))
if not phisat_errors.empty:
    display(phisat_errors.head(DISPLAY_ROWS))


In [ ]:
def query_s2_candidates_for_phisat(phisat_row: pd.Series) -> Tuple[pd.DataFrame, Optional[dict[str, Any]]]:
    phisat_geom = parse_footprint(phisat_row["GeoFootprint"])
    phisat_start = pd.to_datetime(phisat_row["phisat2_start_time"], utc=True)
    start_date, end_date = time_window(phisat_start, MAX_TIME_DELTA_DAYS)

    searcher = CopernicusDataSearcher()
    searcher.query_by_filter(
        collection_name=S2_COLLECTION_NAME,
        product_type=S2_PRODUCT_TYPE,
        aoi_wkt=phisat_geom.wkt,
        start_date=start_date,
        end_date=end_date,
        top=S2_RESULTS_PER_PRODUCT,
        cloud_cover_threshold=S2_CLOUD_COVER_THRESHOLD,
        order_by="ContentDate/Start asc",
    )
    s2_df = searcher.execute_query().copy()
    if s2_df.empty:
        return pd.DataFrame(), {
            "phisat2_list_name": phisat_row["phisat2_list_name"],
            "stage": "s2_candidates",
            "status": "no_s2_candidates_within_time_cloud_window",
            "start_date": start_date,
            "end_date": end_date,
        }

    candidate_rows: list[dict[str, Any]] = []
    for _, s2_row in s2_df.iterrows():
        s2_geom = parse_footprint(s2_row["GeoFootprint"])
        s2_start = content_time(s2_row["ContentDate"], "Start")
        s2_end = content_time(s2_row["ContentDate"], "End")
        delta_seconds = (s2_start - phisat_start).total_seconds()
        intersection_geom = phisat_geom.intersection(s2_geom)
        overlap = overlap_fraction(phisat_geom, s2_geom)
        candidate_rows.append({
            "phisat2_list_name": phisat_row["phisat2_list_name"],
            "phisat2_name": phisat_row.get("Name"),
            "phisat2_id": phisat_row.get("Id"),
            "phisat2_start_time": phisat_start.isoformat(),
            "phisat2_end_time": pd.to_datetime(phisat_row["phisat2_end_time"], utc=True).isoformat(),
            "phisat2_geofootprint": geometry_to_geojson(phisat_geom),
            "s2_name": s2_row.get("Name"),
            "s2_id": s2_row.get("Id"),
            "s2_s3path": s2_row.get("S3Path"),
            "s2_start_time": s2_start.isoformat(),
            "s2_end_time": s2_end.isoformat(),
            "time_delta_seconds": delta_seconds,
            "time_delta_days": delta_seconds / 86400.0,
            "abs_time_delta_days": abs(delta_seconds) / 86400.0,
            "overlap_fraction": overlap,
            "overlap_percent": overlap * 100.0,
            "s2_cloud_cover": attribute_value(s2_row, "cloudCover"),
            "s2_catalogue_coverage_percent": s2_row.get("coverage"),
            "s2_geofootprint": geometry_to_geojson(s2_geom),
            "intersection_geofootprint": geometry_to_geojson(intersection_geom),
            "s2_query_start_date": start_date,
            "s2_query_end_date": end_date,
        })
    return pd.DataFrame(candidate_rows), None


if S2_CANDIDATES_CACHE.exists() and not REFRESH_S2_CANDIDATES:
    s2_candidates_df = read_jsonl(S2_CANDIDATES_CACHE)
    s2_candidate_errors = pd.read_csv(S2_CANDIDATE_ERRORS) if S2_CANDIDATE_ERRORS.exists() else pd.DataFrame()
    print(f"Loaded cached Sentinel-2 candidate rows: {len(s2_candidates_df)}")
else:
    candidate_frames: list[pd.DataFrame] = []
    candidate_errors: list[dict[str, Any]] = []
    for _, phisat_row in tqdm(phisat_df.iterrows(), total=len(phisat_df), desc="Searching Sentinel-2 L1C"):
        try:
            candidates, error = query_s2_candidates_for_phisat(phisat_row)
            if not candidates.empty:
                candidate_frames.append(candidates)
            if error is not None:
                candidate_errors.append(error)
        except Exception as exc:
            candidate_errors.append({
                "phisat2_list_name": phisat_row["phisat2_list_name"],
                "stage": "s2_candidates",
                "status": type(exc).__name__,
                "message": str(exc),
            })
        if REQUEST_SLEEP_SECONDS:
            time.sleep(REQUEST_SLEEP_SECONDS)

    s2_candidates_df = pd.concat(candidate_frames, ignore_index=True) if candidate_frames else pd.DataFrame()
    s2_candidate_errors = pd.DataFrame(candidate_errors)
    write_jsonl(s2_candidates_df, S2_CANDIDATES_CACHE)
    s2_candidate_errors.to_csv(S2_CANDIDATE_ERRORS, index=False)

print(f"Sentinel-2 candidate rows: {len(s2_candidates_df)}")
if not s2_candidates_df.empty:
    print(f"PhiSat-2 products with at least one Sentinel-2 candidate: {s2_candidates_df['phisat2_list_name'].nunique()}")
    display(s2_candidates_df.head(DISPLAY_ROWS))
print(f"Sentinel-2 candidate errors: {len(s2_candidate_errors)}")
if not s2_candidate_errors.empty:
    display(s2_candidate_errors.head(DISPLAY_ROWS))


In [ ]:
def build_coupling_geodataframe(candidates: pd.DataFrame) -> Tuple[gpd.GeoDataFrame, pd.DataFrame, pd.DataFrame]:
    if candidates.empty:
        empty = gpd.GeoDataFrame(geometry=gpd.GeoSeries([], crs="EPSG:4326"))
        return empty, candidates.copy(), candidates.copy()

    prepared = candidates.copy()
    prepared["phisat2_start_time"] = pd.to_datetime(prepared["phisat2_start_time"], utc=True)
    prepared["phisat2_end_time"] = pd.to_datetime(prepared["phisat2_end_time"], utc=True)
    prepared["s2_start_time"] = pd.to_datetime(prepared["s2_start_time"], utc=True)
    prepared["s2_end_time"] = pd.to_datetime(prepared["s2_end_time"], utc=True)
    prepared["s2_cloud_cover"] = pd.to_numeric(prepared["s2_cloud_cover"], errors="coerce")
    prepared["overlap_fraction"] = pd.to_numeric(prepared["overlap_fraction"], errors="coerce")
    prepared["abs_time_delta_days"] = pd.to_numeric(prepared["abs_time_delta_days"], errors="coerce")

    valid = prepared[
        prepared["overlap_fraction"].ge(MIN_SPATIAL_OVERLAP)
        & prepared["abs_time_delta_days"].le(MAX_TIME_DELTA_DAYS)
        & prepared["s2_cloud_cover"].lt(S2_CLOUD_COVER_THRESHOLD)
    ].copy()

    if valid.empty:
        empty = gpd.GeoDataFrame(geometry=gpd.GeoSeries([], crs="EPSG:4326"))
        return empty, prepared, valid

    valid = valid.sort_values(
        ["phisat2_list_name", "abs_time_delta_days", "s2_cloud_cover", "overlap_fraction"],
        ascending=[True, True, True, False],
    )
    best = valid.groupby("phisat2_list_name", sort=False, as_index=False).head(1).reset_index(drop=True)
    best["time_delta"] = pd.to_timedelta(best["time_delta_seconds"], unit="s")
    best["phisat2_geometry"] = best["phisat2_geofootprint"].apply(parse_footprint)
    best["s2_geometry"] = best["s2_geofootprint"].apply(parse_footprint)
    best["intersection_geometry"] = best["intersection_geofootprint"].apply(parse_optional_footprint)

    coupling = gpd.GeoDataFrame(best, geometry="phisat2_geometry", crs="EPSG:4326")
    coupling["s2_geometry"] = gpd.GeoSeries(coupling["s2_geometry"], crs="EPSG:4326")
    coupling["intersection_geometry"] = gpd.GeoSeries(coupling["intersection_geometry"], crs="EPSG:4326")
    return coupling, prepared, valid


coupling_gdf, all_candidates_df, valid_candidates_df = build_coupling_geodataframe(s2_candidates_df)

candidate_summary = pd.DataFrame({"phisat2_list_name": product_names})
metadata_names = set(phisat_df["phisat2_list_name"])
candidate_names = set(all_candidates_df["phisat2_list_name"]) if not all_candidates_df.empty else set()
valid_names = set(valid_candidates_df["phisat2_list_name"]) if not valid_candidates_df.empty else set()
matched_names = set(coupling_gdf["phisat2_list_name"]) if not coupling_gdf.empty else set()

candidate_summary["metadata_found"] = candidate_summary["phisat2_list_name"].isin(metadata_names)
candidate_summary["s2_candidate_found"] = candidate_summary["phisat2_list_name"].isin(candidate_names)
candidate_summary["passes_gates"] = candidate_summary["phisat2_list_name"].isin(valid_names)
candidate_summary["coupled"] = candidate_summary["phisat2_list_name"].isin(matched_names)

if not all_candidates_df.empty:
    metrics = all_candidates_df.groupby("phisat2_list_name", as_index=False).agg(
        candidate_count=("s2_name", "count"),
        max_overlap_fraction=("overlap_fraction", "max"),
        min_abs_time_delta_days=("abs_time_delta_days", "min"),
        min_s2_cloud_cover=("s2_cloud_cover", "min"),
    )
    candidate_summary = candidate_summary.merge(metrics, on="phisat2_list_name", how="left")


def match_status(row: pd.Series) -> str:
    if not row["metadata_found"]:
        return "missing_phisat2_metadata"
    if not row["s2_candidate_found"]:
        return "no_s2_candidates_within_time_cloud_window"
    if not row["passes_gates"]:
        return "no_s2_candidate_passed_cloud_time_overlap_gates"
    return "coupled"


match_log = candidate_summary.copy()
match_log["match_status"] = match_log.apply(match_status, axis=1)
match_log.to_csv(MATCH_LOG_CSV, index=False)

print(f"Coupled products: {len(coupling_gdf)} / {len(product_names)}")
display(match_log["match_status"].value_counts().rename_axis("match_status").reset_index(name="count"))


In [ ]:
if not coupling_gdf.empty:
    display_columns = [
        "phisat2_list_name",
        "phisat2_name",
        "s2_name",
        "phisat2_start_time",
        "s2_start_time",
        "time_delta",
        "time_delta_days",
        "overlap_percent",
        "s2_cloud_cover",
        "s2_s3path",
    ]
    display(coupling_gdf[[column for column in display_columns if column in coupling_gdf.columns]].head(DISPLAY_ROWS))
else:
    print("No PhiSat-2 / Sentinel-2 L1C pairs passed the configured gates.")

# Lightweight exports for downstream inspection.
if not coupling_gdf.empty:
    geojson_gdf = coupling_gdf.drop(columns=["s2_geometry", "intersection_geometry"], errors="ignore").copy()
    for column in geojson_gdf.select_dtypes(include=["datetimetz", "datetime64[ns]"]).columns:
        geojson_gdf[column] = geojson_gdf[column].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    COUPLING_GEOJSON.write_text(geojson_gdf.to_json(default=str), encoding="utf-8")

    csv_df = coupling_gdf.drop(
        columns=["phisat2_geometry", "s2_geometry", "intersection_geometry"],
        errors="ignore",
    ).copy()
    csv_df["phisat2_geometry_wkt"] = coupling_gdf["phisat2_geometry"].map(lambda geom: geom.wkt if geom is not None else None)
    csv_df["s2_geometry_wkt"] = coupling_gdf["s2_geometry"].map(lambda geom: geom.wkt if geom is not None else None)
    csv_df["intersection_geometry_wkt"] = coupling_gdf["intersection_geometry"].map(lambda geom: geom.wkt if geom is not None else None)
    csv_df.to_csv(COUPLING_CSV, index=False)

print(f"Match log: {MATCH_LOG_CSV}")
if COUPLING_GEOJSON.exists():
    print(f"Coupling GeoJSON: {COUPLING_GEOJSON}")
if COUPLING_CSV.exists():
    print(f"Coupling CSV: {COUPLING_CSV}")


In [ ]:
# Optional quick spatial check for the first matches.
# The active geometry is the PhiSat-2 footprint; Sentinel-2 boundaries are overlaid in green.
if not coupling_gdf.empty:
    sample = coupling_gdf.head(min(20, len(coupling_gdf)))
    ax = sample.boundary.plot(figsize=(8, 8), color="#1f77b4", linewidth=1.5)
    gpd.GeoDataFrame(sample.drop(columns="phisat2_geometry"), geometry="s2_geometry", crs="EPSG:4326").boundary.plot(
        ax=ax,
        color="#2ca02c",
        linewidth=1.0,
        alpha=0.8,
    )
    ax.set_title("PhiSat-2 footprints (blue) and matched Sentinel-2 L1C footprints (green)")
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
